[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# 实操 2：反向传播的简单实现

[视频时间戳](https://youtu.be/Z6H3zakmn6E?t=2640)

这里我们用 `numpy` 为下面的问题实现一个简单的反向传播算法：

我们生成点 $(x_t,y_t)$，其中 $y_t= \exp(w^*x_t+b^*)$，也就是说 $y^*_t$ 是把参数为 $w^*$ 和 $b^*$ 的确定性函数作用在 $x_t$ 上得到的。我们的目标是根据观测值 $(x_t,y_t)$ 恢复出参数 $w^*$ 和 $b^*$。

为此，我们用 SGD 关于 $w$ 和 $b$ 最小化 $\sum_t(y^t - \exp(w x_t+b))^2$。

在这次的实操中，我们实现的是**小批次大小为 1 的随机梯度下降（SGD）**，而不是第二课里讲的批量梯度下降。

算法的改动如下：我们要最小化的损失是：
$$
loss = \sum_t\underbrace{\left(\exp(w x_t+b)-y_t \right)^2}_{loss_t}.
$$

为了最小化损失，我们先计算每个 $loss_t$ 的梯度：$\frac{\partial{loss_t}}{\partial w}$ 和 $\frac{\partial{loss_t}}{\partial b}$。

在一个 epoch 里，**小批次大小为 1 的随机梯度下降**通过运行下面的循环来更新权重和偏置：

对 $t \in \{1,\dots,30\}$，

\begin{eqnarray*}
w_{t+1}&=&w^1_{t}-\alpha\frac{\partial{loss_t}}{\partial w} \\
b_{t+1}&=&b_{t}-\alpha\frac{\partial{loss_t}}{\partial b},
\end{eqnarray*}

如果 $t = 30$，就令 $w_1=w_{31}$，$b_1=b_{31}$。

$\alpha>0$ 叫做学习率。

然后跑多个 epoch……

可以看到，一个 epoch 的代价比批量梯度下降小得多，因为只需要一个样本就能做一次更新；而在批量设定下，做一次更新需要遍历整个数据集。当然，这里我们算的并不是真正的梯度，那些偏导数可以看作梯度的带噪估计。


In [ ]:
# 这次实操不需要 PyTorch
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
w, b = 0.5, 2
xx = np.arange(0,1,.01)
yy = np.exp(w*xx+b)

In [ ]:
plt.plot(yy)

按照课程里刚讲的内容，你需要实现每个基本运算：`(.*w), (.+b), exp(.)`，每个运算都要有 forward、backward 和 step 方法。


如果你还不熟悉 Python 的[类](https://docs.python.org/3/tutorial/classes.html)，应该先学一下基础，因为后面的课程都会用到。

为了帮你上手，下面我已经把 `(.+b)` 运算实现成了一个 Python 类：


In [ ]:
class add_bias(object):
    def __init__(self,b):
        # 用一个偏置 b 初始化
        self.b = b
        
    def forward(self, x):
        # 返回加上偏置的结果
        return x + self.b
    
    def backward(self,grad):
        # 保存梯度（供 step 方法更新偏置用）并返回反向传播的梯度
        self.grad = grad
        return grad
    
    def step(self, learning_rate):
        # 更新偏置
        self.b -= learning_rate*self.grad        

现在考虑一个更简单的问题：假设 $z_t = x_t+b^*$，你的任务是用 SGD 关于 $b$ 最小化损失 $\sum_t(x_t+b-z_t)^2$，从而估计出 $b^*$。你可以像下面这样使用上面定义的 `add_bias`：


In [ ]:
# 先用真实的偏置 5 计算 z_t：
zz = xx+5

#从偏置的初始猜测值 1 开始：
My_add_bias = add_bias(1)

In [ ]:
j = 10
# 你的预测将对每个样本给出
z_pred = My_add_bias.forward(xx[j])
z_pred

In [ ]:
# 从二次损失的梯度开始
grad = 2*(z_pred-zz[j])

In [ ]:
# 把梯度反向传播到参数 b
My_add_bias.backward(grad)

In [ ]:
# 对偏置做一次更新
My_add_bias.step(1e-2)
My_add_bias.b

上面的代码对应一次 SGD 更新。
下面我写好了 SGD 的训练循环，每看到一个样本就更新一次参数：对每个样本 $j$，计算它对应的


In [ ]:
My_add_bias = add_bias(1)
estimated_b = [1]
for i in range(500):
    # 随机取一个索引
    j = np.random.randint(1, len(xx))
    z_pred = My_add_bias.forward(xx[j])
    grad = 2*(z_pred-zz[j])
    _ = My_add_bias.backward(grad)
    My_add_bias.step(1e-2)
    estimated_b.append(My_add_bias.b)

In [ ]:
plt.plot(estimated_b)

虽然 SGD 算的是梯度的带噪版本，但在这个例子里我们看到 SGD 依然收敛到了正确的解。

现在轮到你了！
按照课程里刚讲的内容，你需要实现每个基本运算：`(.*w), exp(.)`，每个运算都要有 forward、backward 和 step 方法。

![backprop3](https://dataflowr.github.io/notebooks/Module2/img/backprop3.png)


In [ ]:
class multiplication_weight(object):
    def __init__(self, w):
        # 用一个权重 w 初始化
        
    def forward(self, x):
        # 返回乘以权重后的结果
               
    def backward(self,grad):
        # 保存梯度并返回反向传播的梯度
            
    def step(self, learning_rate):
        # 更新权重
        
class my_exp(object):
    # 没有参数
    def forward(self, x):
        # 返回 exp(x)
            
    def backward(self,grad):
        # 返回反向传播的梯度
            
    def step(self, learning_rate):
        # 有要更新的参数吗？
        # 提示：https://docs.python.org/3/reference/simple_stmts.html#the-pass-statement
        

现在，你需要把这些运算按顺序组合起来，这里要自己写一个组合运算的类。这个类要有 forward、backward、step 方法，还要有一个 compute_loss 方法。


In [ ]:
class my_composition(object):
    def __init__(self, layers):
        # 按正确的顺序用所有运算（这里叫 layers！）初始化……
                
    def forward(self, x):
        # 依次应用每个层的 forward 方法
            
    def compute_loss(self, y_est, y_target):
        # 使用 L2 损失
        # 返回损失并保存损失的梯度
            
    def backward(self):
        # 从损失的梯度开始，按顺序反向传播
        # 提示：https://docs.python.org/3/library/functions.html#reversed
            
    def step(self, learning_rate):
        # 依次应用每个层的 step 方法
        

现在你需要写'训练'循环。记录每个 epoch 计算出的损失、权重和偏置。


In [ ]:
my_fit = my_composition([multiplication_weight(1),add_bias(1), my_exp()])
learning_rate = 1e-4
losses =[]
ws = []
bs = []
for i in range(5000):
    # 随机取一个索引
    j = np.random.randint(1, len(xx))
    # 你可以和下面这行对比
    #j = i % len(xx)
    # 用当前参数值从 xx[j] 计算 y 的估计值
    
    # 计算损失并保存
    
    # 更新参数
    
    #并保存它们
    ws.append(my_fit.layers[0].w)
    bs.append(my_fit.layers[1].b)

In [ ]:
my_fit.layers[0].w

In [ ]:
my_fit.layers[1].b

In [ ]:
plt.plot(losses)

In [ ]:
plt.plot(bs)

In [ ]:
plt.plot(ws)

现在你就明白 PyTorch 是怎么处理自动微分的了！！


[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)